# STRATIFIED K-FOLD

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc # garbage collector to help free memory
import wandb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold 
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# configuration

EXCEL_PATH    = "Database EcoTSA-Coronaropatia.xlsx"
TEXT_COLUMN   = "REFERTO"
LABEL_COLUMN  = "CORONAROPATIA"

MODEL_NAME = "dbmdz/bert-base-italian-cased"

MAX_LEN    = 512
BATCH_SIZE = 16
RANDOM_SEED = 42

EPOCHS_PHASE2 = 5
LR_PHASE2     = 2e-5  

In [ ]:
# load data

df = pd.read_excel(EXCEL_PATH, header=1)
df = df.dropna(subset=[LABEL_COLUMN, TEXT_COLUMN])

# force the label column to be an integer (0 or 1)
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(int)

print(f"Cleaned dataset shape: {df.shape}")
print(f"\nLabel distribution:\n{df[LABEL_COLUMN].value_counts()}\n")

In [ ]:
# tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# dataset helper functions
def make_dataset(df_split):
    return Dataset.from_pandas(
        df_split[[TEXT_COLUMN, LABEL_COLUMN]].rename(
            columns={TEXT_COLUMN: "text", LABEL_COLUMN: "label"}
        ).reset_index(drop=True)
    )

def tokenize(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

# grading rubric for the Trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

In [ ]:
# initialize the K-Fold tool
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# empty lists to store the accuracies, graphs, and the master pool of data
fold_accuracies = []
all_diaries = []
all_eval_dfs = [] # pooling of the 5 folds

class FoldWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=weights)(outputs.logits, labels.long())
        return (loss, outputs) if return_outputs else loss

print("── Starting 5-Fold Stratified Cross-Validation ──\n")

for fold, (train_idx, eval_idx) in enumerate(skf.split(df, df[LABEL_COLUMN])):
    
    print(f"\n========== FOLD {fold + 1} ==========")

    train_df = df.iloc[train_idx].copy()
    eval_df = df.iloc[eval_idx].copy()
    
    print(f"Train size: {len(train_df)} | Eval size: {len(eval_df)}")

    # recalculate class weights for this fold
    weights_array = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(train_df[LABEL_COLUMN]),
        y=train_df[LABEL_COLUMN]
    )
    class_weight_dict = dict(enumerate(weights_array))
    weights = torch.tensor([class_weight_dict[i] for i in sorted(class_weight_dict)], dtype=torch.float).to(device)

    # dataset prep
    dataset_train = make_dataset(train_df).map(tokenize, batched=True)
    dataset_eval  = make_dataset(eval_df).map(tokenize, batched=True)

    dataset_train = dataset_train.map(lambda x: {"label": int(x["label"])})
    dataset_eval  = dataset_eval.map(lambda x: {"label": int(x["label"])})

    # load a fresh model
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    model.to(device)

    # start wandb
    wandb.init(
        project="tesi-magistrale",   
        name=f"kfold_run_fold_{fold + 1}", 
        reinit=True
    )

    training_args = TrainingArguments(
        output_dir=f"./run2_results_fold_{fold + 1}", 
        num_train_epochs=EPOCHS_PHASE2, 
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR_PHASE2,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="wandb",
        weight_decay=0.0,
        warmup_steps=0,
        lr_scheduler_type="constant",
    )

    trainer = FoldWeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset_train,
        eval_dataset=dataset_eval,
        compute_metrics=compute_metrics,
    )

    print(f"Training Fold {fold + 1}...")
    trainer.train()

    print(f"Evaluating Fold {fold + 1}...")
    predictions = trainer.predict(dataset_eval)
    preds = np.argmax(predictions.predictions, axis=1)
    labels = predictions.label_ids

    acc = accuracy_score(labels, preds)
    fold_accuracies.append(acc)

    print(f"Fold {fold + 1} Final Accuracy: {acc:.4f}\n")
    
    # save the diary for the graph
    all_diaries.append(trainer.state.log_history)

    # --- SAVE TO THE MASTER POOL ---
    eval_df = eval_df.copy()
    eval_df["model_prediction"] = preds
    eval_df["bert_token_count"] = eval_df[TEXT_COLUMN].apply(
        lambda x: len(tokenizer.tokenize(str(x)))
    )
    eval_df["is_correct"] = eval_df[LABEL_COLUMN] == eval_df["model_prediction"]
    eval_df["fold"] = fold + 1
    
    all_eval_dfs.append(eval_df) 

    # system cleanup
    del model
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    
    wandb.finish()

print("\n========== K-Fold Cross-Validation ==========")
print(f"Accuracies across 5 folds: {[round(num, 4) for num in fold_accuracies]}")
mean_acc = np.mean(fold_accuracies)
std_acc = np.std(fold_accuracies)
print(f"AVERAGE ACCURACY: {mean_acc:.4f} ± {std_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

# start final wandb session 
wandb.init(project="tesi-magistrale", name="kfold_final_pooled_results")

# average graph

train_loss_sums = defaultdict(float)
train_loss_counts = defaultdict(int)
eval_loss_sums = defaultdict(float)
eval_loss_counts = defaultdict(int)

for diary in all_diaries:
    for entry in diary:
        if 'epoch' in entry:
            epoch = int(entry['epoch'])
            if 'loss' in entry:
                train_loss_sums[epoch] += entry['loss']
                train_loss_counts[epoch] += 1
            elif 'eval_loss' in entry:
                eval_loss_sums[epoch] += entry['eval_loss']
                eval_loss_counts[epoch] += 1

epochs_train = sorted(train_loss_sums.keys())
avg_train_loss = [train_loss_sums[e] / train_loss_counts[e] for e in epochs_train]

epochs_eval = sorted(eval_loss_sums.keys())
avg_eval_loss = [eval_loss_sums[e] / eval_loss_counts[e] for e in epochs_eval]

plt.figure(figsize=(8, 5))
plt.plot(epochs_train, avg_train_loss, label='Average Train Loss', color='blue', marker='o')
plt.plot(epochs_eval, avg_eval_loss, label='Average Validation Loss', color='orange', marker='o')
plt.title('Average Model Loss (5-Fold Cross-Validation)')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.xticks(epochs_train)
plt.legend(loc='upper right')
plt.grid(True)
plt.tight_layout()

wandb.log({"average_loss_graph": wandb.Image(plt)})

plt.show() 

# overall pooled metrics table

print("\n========================================================")
print("    Final K-Fold Metrics    ")
print("========================================================\n")

master_df = pd.concat(all_eval_dfs)
y_true_all = master_df[LABEL_COLUMN]
y_pred_all = master_df['model_prediction']

print(f"Overall Accuracy: {accuracy_score(y_true_all, y_pred_all):.4f}\n")
print("Classification Report:")
print(classification_report(y_true_all, y_pred_all, target_names=["Negative", "Positive"]))
print("Confusion Matrix:")
print(confusion_matrix(y_true_all, y_pred_all))

# upload to wandb
report_dict = classification_report(y_true_all, y_pred_all, target_names=["Negative", "Positive"], output_dict=True)
wandb.log({"pooled_overall_report": wandb.Table(dataframe=pd.DataFrame(report_dict).transpose())})

# truncated reports analysis

print("\n========== Truncated Reports Analysis ==========\n")

truncated_df = master_df[master_df['bert_token_count'] > 512]

if not truncated_df.empty:
    print(f"Total Truncated Reports across all 5 folds: {len(truncated_df)}\n")
    print(truncated_df[['fold', 'model_prediction', LABEL_COLUMN, 'is_correct', 'bert_token_count']])
    
    y_true_trunc = truncated_df[LABEL_COLUMN]
    y_pred_trunc = truncated_df['model_prediction']
    
    print(f"\nAccuracy on Truncated Reports ONLY: {accuracy_score(y_true_trunc, y_pred_trunc):.4f}\n")
    print("Classification Report (Truncated):")
    print(classification_report(y_true_trunc, y_pred_trunc, target_names=["Negative", "Positive"]))
    print("Confusion Matrix (Truncated):")
    print(confusion_matrix(y_true_trunc, y_pred_trunc))

    # upload to wandb
    trunc_dict = classification_report(y_true_trunc, y_pred_trunc, target_names=["Negative", "Positive"], output_dict=True)
    wandb.log({"truncated_reports_report": wandb.Table(dataframe=pd.DataFrame(trunc_dict).transpose())})

else:
    print("No truncated reports found.")

wandb.finish()